In [0]:
"""
Module: Gold Configuration
Description: Sets up the catalog and enables Spark SQL features for 
             high-performance aggregation.
"""
dbutils.widgets.text("catalog_name", "nyc_taxi_dev", "Target Catalog")
CATALOG = dbutils.widgets.get("catalog_name")

print(f"🌟 Gold Layer initialized for: {CATALOG}")

In [0]:
"""
Module: Task D1 - Daily Performance Aggregates
Description: Aggregates taxi metrics (revenue, trip count) by date 
             and joins with weather conditions for business reporting.
Target: {CATALOG}.gold.fact_daily_taxi_performance
"""
from pyspark.sql.functions import col, count, sum, avg, to_date

df_enriched = spark.read.table(f"{CATALOG}.silver.enriched_taxi_weather")

# Aggregate metrics by date and weather context
df_gold_daily = df_enriched.groupBy(
    to_date("pickup_time").alias("date"),
    "precipitation_mm",
    "temp_max",
    "avg_temp"
).agg(
    count("trip_hash_id").alias("total_trips"),
    sum("total_amount").alias("total_revenue"),
    avg("trip_distance").alias("avg_distance"),
    avg("fare_amount").alias("avg_base_fare")
).orderBy("date")

# Write to Gold (Business users usually prefer Overwrite for daily snapshots)
df_gold_daily.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.gold.fact_daily_taxi_performance")

print("✅ Gold: Daily Performance table created.")

In [0]:
"""
Module: Task D2 - Weather Category Analysis
Description: Buckets precipitation into categorical groups to analyze 
             how different weather "types" impact business KPIs.
Target: {CATALOG}.gold.rpt_weather_impact
"""
from pyspark.sql.functions import when, col, avg

df_daily = spark.read.table(f"{CATALOG}.gold.fact_daily_taxi_performance")

# Define business logic for weather categories
df_impact = df_daily.withColumn(
    "weather_category",
    when(col("precipitation_mm") == 0, "Clear")
    .when(col("precipitation_mm") < 5, "Light Rain")
    .otherwise("Heavy Rain")
).groupBy("weather_category").agg(
    avg("total_trips").alias("avg_daily_trips"),
    avg("total_revenue").alias("avg_daily_revenue"),
    avg("avg_base_fare").alias("avg_fare_per_trip")
)

df_impact.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.gold.rpt_weather_impact")

print("🏆 Gold: Weather Impact Report ready for visualization.")